In [0]:
# ==========================================
# Project: Retail Medallion Architecture
# Layer: Silver
# Purpose: Clean and enrich Bronze data
# ==========================================

from pyspark.sql import functions as F

# ------------------------------------------
# Configuration
# ------------------------------------------

BRONZE_TABLE = "retail_project.bronze.online_retail"
SILVER_TABLE = "retail_project.silver.online_retail"

# ------------------------------------------
# Read Bronze Data
# ------------------------------------------

df_bronze = spark.table(BRONZE_TABLE)

print("Bronze Count:", df_bronze.count())

# ------------------------------------------
# Data Cleaning & Enrichment
# ------------------------------------------

df_silver = (
    df_bronze
    .dropDuplicates()
    .filter(F.col("Description").isNotNull())
    .filter(F.col("UnitPrice") > 0)
    .withColumn(
        "transaction_type",
        F.when(
            F.col("InvoiceNo").startswith("C"),
            "RETURN"
        ).otherwise("SALE")
    )
)

# ------------------------------------------
# Validation
# ------------------------------------------

print("Silver Count:", df_silver.count())

df_silver.groupBy("transaction_type").count().show()

# ------------------------------------------
# Write Silver Table
# ------------------------------------------

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

# ------------------------------------------
# Post-Load Validation
# ------------------------------------------

spark.sql(f"""
SELECT COUNT(*) AS row_count
FROM {SILVER_TABLE}
""").show()

print("Silver transformation completed successfully.")